In [ ]:
# !pip install pennylane sentence-transformers autograd torch numpy

In [2]:
import pennylane as qml
import autograd.numpy as np
from autograd import grad
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer
import numpy as regular_np
import warnings
warnings.filterwarnings('ignore')

c:\Users\Emmanuel.DESKTOP-V1AHMTP\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Dataset
train_texts = [
    # Positive reviews (30 total)
    "This movie is great and amazing",
    "I love this film so much",
    "The acting was wonderful",
    "Absolutely brilliant and entertaining",
    "A masterpiece of modern cinema",
    "The best movie I have seen this year",
    "Incredible performances by all actors",
    "This film exceeded all my expectations",
    "A truly captivating and emotional story",
    "Outstanding direction and beautiful cinematography",
    "I was thoroughly impressed and moved",
    "An absolute gem that everyone should watch",
    "Fantastic storytelling and great pacing",
    "The plot was engaging from start to finish",
    "A delightful and heartwarming experience",
    "Visually stunning with powerful performances",
    "This film touched my heart deeply",
    "A refreshing and original take on the genre",
    "The chemistry between the leads was perfect",
    "Brilliantly crafted with attention to detail",
    "Every scene was beautifully executed",
    "A thrilling ride from beginning to end",
    "The soundtrack perfectly complemented the story",
    "Witty dialogue and memorable characters",
    "I laughed and cried throughout this film",
    "A triumph of storytelling and vision",
    "The cinematography was breathtaking",
    "Perfectly paced with no dull moments",
    "An inspiring and uplifting masterpiece",
    "This deserves all the awards and praise",

    # Negative reviews (30 total)
    "This is terrible and boring",
    "I hate this movie completely",
    "Worst film I have ever seen",
    "Absolutely awful and disappointing",
    "A complete waste of time and money",
    "The acting was terrible throughout",
    "This movie was painfully boring",
    "I regret watching this disaster",
    "Poorly written and badly directed",
    "The worst performance I have ever witnessed",
    "Utterly predictable and unoriginal",
    "This film was a total disappointment",
    "Boring plot with terrible dialogue",
    "I could not wait for this to end",
    "Absolutely unwatchable and dreadful",
    "The plot made absolutely no sense",
    "Terrible pacing that dragged on forever",
    "Wooden acting and unconvincing performances",
    "A formulaic and cliched mess",
    "The special effects looked cheap and fake",
    "I was bored within the first ten minutes",
    "Confusing storyline with plot holes everywhere",
    "The characters were flat and unlikeable",
    "A pretentious and self indulgent failure",
    "Poorly edited with jarring transitions",
    "The dialogue was cringe worthy and forced",
    "Completely forgettable and generic",
    "Terrible casting choices ruined it",
    "This film insulted my intelligence",
    "A sloppy and amateurish production"
]

train_labels = [
    # Positive labels (30 total)
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
    # Negative labels (30 total)
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0
]

test_texts = [
    # Positive reviews (10 total)
    "This is an excellent movie",
    "A remarkable film with stunning visuals",
    "I thoroughly enjoyed every minute of it",
    "The story was compelling and well crafted",
    "Simply amazing and highly recommended",
    "A cinematic experience that stays with you",
    "Superb acting and directing throughout",
    "This movie raised the bar for the genre",
    "An emotionally powerful and beautiful film",
    "Expertly crafted with brilliant execution",

    # Negative reviews (10 total)
    "I really dislike this film",
    "This was a boring and forgettable movie",
    "The script was weak and unconvincing",
    "I found this film extremely disappointing",
    "A poor attempt at filmmaking",
    "Dull and lifeless from start to finish",
    "The acting felt forced and artificial",
    "A tedious experience that went nowhere",
    "Poorly conceived and badly executed",
    "This movie completely missed the mark"
]

test_labels = [
    # Positive labels (10 total)
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
    # Negative labels (10 total)
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0
]

print(f"\nDataset: {len(train_texts)} train, {len(test_texts)} test samples")



Dataset: 60 train, 20 test samples


In [4]:
# Load and encode with SLM
print("Loading SLM...")
slm = SentenceTransformer('paraphrase-MiniLM-L3-v2') # from sentence to vectors of 38 elements
train_embeddings = slm.encode(train_texts, show_progress_bar=True) # 384 vector size
test_embeddings = slm.encode(test_texts, show_progress_bar=True) # # 384 vector size

Loading SLM...


Batches: 100%|██████████| 1/1 [00:00<00:00, 78.39it/s]


In [5]:
print("\nReducing dimensions for quantum circuit\n")

n_qubits = 6
reducer = nn.Linear(384, n_qubits)

with torch.no_grad():
    train_reduced = reducer(torch.tensor(train_embeddings, dtype=torch.float32)).numpy() # from 384 to n_qubits size
    test_reduced = reducer(torch.tensor(test_embeddings, dtype=torch.float32)).numpy() # from 384 to n_qubits size

train_reduced = np.clip(np.array(train_reduced, dtype=np.float64), -np.pi, np.pi) # for angle embedding
test_reduced = np.clip(np.array(test_reduced, dtype=np.float64), -np.pi, np.pi) # for angle embedding

print(f"Reduced to {n_qubits} features")


Reducing dimensions for quantum circuit

Reduced to 6 features


In [6]:
# Quantum circuit
dev = qml.device('lightning.qubit', wires=n_qubits)

@qml.qnode(dev, interface='autograd')
def quantum_circuit(inputs, weights):
    n_layers = weights.shape[0]

    # Initial encoding (angle embedding)
    for i in range(n_qubits):
        qml.RX(inputs[i], wires=i)
        qml.RY(inputs[i], wires=i)

    # Variational rotations
    for layer in range(n_layers):
        for i in range(n_qubits):
            qml.RX(weights[layer, i, 0], wires=i)
            qml.RY(weights[layer, i, 1], wires=i)
            qml.RZ(weights[layer, i, 2], wires=i)

        # Entangling
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[n_qubits - 1, 0])

        if layer < n_layers - 1:
            for i in range(n_qubits):
                qml.RX(inputs[i], wires=i)
                qml.RY(inputs[i], wires=i)

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [7]:
# Initialize parameters
n_layers = 4
quantum_weights = np.array(regular_np.random.randn(n_layers, n_qubits, 3).astype(np.float64) * 0.1)
classical_weights = np.array(regular_np.random.randn(n_qubits).astype(np.float64) * 0.1)
classical_bias = np.float64(0.0)

print(f"Circuit: {n_qubits} qubits, {n_layers} layers, {n_layers * n_qubits * 3 + n_qubits + 1} params")

Circuit: 6 qubits, 4 layers, 79 params


In [8]:
# Hybrid model (Classical Output Layer (Linear regression kind is used))
def full_model(inputs, q_weights, c_weights, c_bias):
    quantum_features = np.array(quantum_circuit(inputs, q_weights))
    logit = np.dot(c_weights, quantum_features) + c_bias
    prediction = 1.0 / (1.0 + np.exp(-logit))
    return prediction, quantum_features

In [9]:
# Loss and gradients
def loss_fn(q_weights, c_weights, c_bias):
    total_loss = 0.0
    for i in range(len(train_reduced)):
        pred, _ = full_model(train_reduced[i], q_weights, c_weights, c_bias)
        label = train_labels[i]
        total_loss += -(label * np.log(pred + 1e-9) + (1 - label) * np.log(1 - pred + 1e-9))
    return total_loss / len(train_reduced)

grad_q = grad(loss_fn, 0)
grad_c_w = grad(loss_fn, 1)
grad_c_b = grad(loss_fn, 2)

In [10]:
# Training
learning_rate = 0.7
n_epochs = 100

print("\nTraining...")
print("Epoch  Loss")

for epoch in range(n_epochs):
    current_loss = loss_fn(quantum_weights, classical_weights, classical_bias)

    grad_quantum = grad_q(quantum_weights, classical_weights, classical_bias)
    grad_classical_w = grad_c_w(quantum_weights, classical_weights, classical_bias)
    grad_classical_b = grad_c_b(quantum_weights, classical_weights, classical_bias)

    quantum_weights = quantum_weights - learning_rate * grad_quantum
    classical_weights = classical_weights - learning_rate * grad_classical_w
    classical_bias = classical_bias - learning_rate * grad_classical_b

    if (epoch + 1) % 10 == 0:
        print(f"{epoch + 1:3d}    {current_loss:.4f}")


Training...
Epoch  Loss
 10    0.6574
 20    0.5162
 30    0.4017
 40    0.3332
 50    0.2885
 60    0.2590
 70    0.2386
 80    0.2235
 90    0.2132
100    0.2482


In [11]:
# Evaluate
print("\nTraining Results:")
train_correct = 0
for i in range(len(train_reduced)):
    pred, _ = full_model(train_reduced[i], quantum_weights, classical_weights, classical_bias)
    pred_val = float(pred)
    pred_label = 1 if pred_val > 0.5 else 0
    train_correct += (pred_label == train_labels[i])

    status = "Correct" if pred_label == train_labels[i] else "Wrong"
    sentiment = "Positive" if pred_label == 1 else "Negative"
    print(f"{i+1}. \"{train_texts[i][:40]}...\" -> {sentiment} ({pred_val:.3f}) {status}")

print(f"\nTrain Accuracy: {100*train_correct/len(train_reduced):.1f}%")

print("\nTest Results:")
test_correct = 0
for i in range(len(test_reduced)):
    pred, q_feat = full_model(test_reduced[i], quantum_weights, classical_weights, classical_bias)
    pred_val = float(pred)
    pred_label = 1 if pred_val > 0.5 else 0
    test_correct += (pred_label == test_labels[i])

    status = "Correct" if pred_label == test_labels[i] else "Wrong"
    sentiment = "Positive" if pred_label == 1 else "Negative"
    q_feat_vals = [float(x) for x in q_feat]

    print(f"{i+1}. \"{test_texts[i][:40]}...\"")
    print(f"   Q-features: {[f'{x:.3f}' for x in q_feat_vals]} -> {sentiment} ({pred_val:.3f}) {status}")

print(f"\nTest Accuracy: {100*test_correct/len(test_reduced):.1f}%")


Training Results:
1. "This movie is great and amazing..." -> Positive (0.865) Correct
2. "I love this film so much..." -> Positive (0.907) Correct
3. "The acting was wonderful..." -> Positive (0.910) Correct
4. "Absolutely brilliant and entertaining..." -> Positive (0.965) Correct
5. "A masterpiece of modern cinema..." -> Positive (0.941) Correct
6. "The best movie I have seen this year..." -> Positive (0.787) Correct
7. "Incredible performances by all actors..." -> Positive (0.963) Correct
8. "This film exceeded all my expectations..." -> Negative (0.438) Wrong
9. "A truly captivating and emotional story..." -> Positive (0.969) Correct
10. "Outstanding direction and beautiful cine..." -> Positive (0.954) Correct
11. "I was thoroughly impressed and moved..." -> Positive (0.614) Correct
12. "An absolute gem that everyone should wat..." -> Positive (0.905) Correct
13. "Fantastic storytelling and great pacing..." -> Positive (0.944) Correct
14. "The plot was engaging from start to fini..